In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
import pandas as pd
import dataprocess.extractor as extractor
import dataprocess.tableutils as tableutils
import utils.utils as utils
import dataprocess.processor as processor
import json
import dataprocess.filters as filters


## Zen-Traffic CF Data Extraction

In [3]:
DATA_FILE = "F:\DATA\Zen2\Zen-traffic\Data"
ROAD_SECTION_NAMES = {1: 11, 2: 4, 3: 13}
ZEN_TIMEFORMAT = "%H%M%S%f"
LAYOUT_FOLDER = ".\data\layouts"
MAINSTREAM = [1, 2] # driving lane and passing lane


data_file = lambda route, sec: os.path.join(DATA_FILE, f"HanshinExpresswayRoute{ROAD_SECTION_NAMES[route]}", "TrafficData", "TRAJECTORY", f"L00{route}_F00{sec}_trajectory.csv")



In [4]:

column_names = ["ID",  # vehicle ID
                "TIME",  # time (originally in str)
                "TYPE",  # vehicle type (1: normal vehicle, 2: large vehicle(bus, truck, etc.))
                "SPD", # speed (kph)
                "LANE", # lane number
                "LAT", # lateral position
                "LON", # longitudinal position
                "KILO", # position (meter)
                "LEN", # vehicle length (meter)
                "OBS"] # observation flag (1: detected record by image recognition, 0: interpolated record)

names_mapping = {
    "TIME": "TIME", 
    "ID": "ID",
    "SPD": "SPD",
    "KILO": "KILO",
    "LANE": "LANE",
    "LC": "LC",
    "ACC": "ACC"
}



In [5]:

route = 1
scene = 1

# load raw data
raw_data = pd.read_csv(data_file(route, scene), names=column_names)

# load route layout
with open(os.path.join(LAYOUT_FOLDER, f"Route{int(ROAD_SECTION_NAMES[route])}.json"), 'r') as layout_file:
        layout = json.load(layout_file) 
        layout["mainstream"] = MAINSTREAM

zen_processor = processor.DataProcessor(rise=False, in_kph=True, time_resolution=0.1)

data = zen_processor.strtime2sec(raw_data, ZEN_TIMEFORMAT) # convert time from str to seconds
data = zen_processor.to_ms(data) # convert kph to m/s
data = zen_processor.fix_rise(data) # fix the KILO to be rising
data = zen_processor.generate_lc(data, 30) # generate lane change (LC) information
data = zen_processor.kalman_filter(data) # apply kalman filter to smooth the trajectory
data = zen_processor.derive_acc(data)
data = zen_processor.set_index(data) # set multi-index (TIME, ID)


print(tableutils.lookup(data, time_slice=(10, 20)))

process_result = zen_processor.get_result(data)



           TYPE        SPD  LANE         LAT        LON         KILO  LEN  \
ID   TIME                                                                   
0    10.0     1  20.931095     2  135.460538  34.719401   194.370452  4.5   
     10.1     1  20.917874     2  135.460542  34.719385   196.230968  4.5   
     10.2     1  20.851265     2  135.460546  34.719368   198.020671  4.5   
     10.3     1  20.720154     2  135.460551  34.719350   199.692420  4.5   
     10.4     1  20.693625     2  135.460555  34.719332   201.402421  4.5   
...         ...        ...   ...         ...        ...          ...  ...   
3673 19.6     1  19.755676     2  135.468427  34.712758  1298.906782  3.5   
     19.7     1  19.805842     2  135.468440  34.712744  1300.944004  3.5   
     19.8     1  19.841016     2  135.468453  34.712729  1302.995816  3.5   
     19.9     1  19.852741     2  135.468466  34.712714  1305.057743  3.5   
     20.0     1  19.831523     2  135.468478  34.712700  1307.088698  3.5   

In [6]:
extract_name_list = ["ID", "TIME", "SPD", "KILO", "LANE", "LC", "ACC"]

def gen_ex(process_result: processor.ProcessResult) -> extractor.VehicleTimeExtractor:
    ex = extractor.VehicleTimeExtractor(process_result, layout, extract_name_list)
    ex._generate_veh_sort_graph() # to speed-up the extraction process, pre-cache the sorted vehicle platoon on each lane.
    return ex

ex = gen_ex(process_result)



100%|██████████| 38761/38761 [05:23<00:00, 119.85it/s]


In [23]:
veh_filter = filters.VehicleTimeFilter(extract_name_list)

self_filter_set = [veh_filter.in_acc_range((-20, 6)), 
                   veh_filter._veh_not_on_lane(3), 
                   veh_filter._veh_exist, 
                   veh_filter.no_lc]

leader_filter_set = [veh_filter.in_acc_range((-20, 6)), 
                     veh_filter._veh_exist]

In [ ]:
include_config = {
    "include_leader": True,
    "include_self": True
}

filter_dict = {
    "self": self_filter_set,
    "leader": leader_filter_set,
}

seg_length = 300 # segment length in frames (30 seconds)

roller = extractor.WindowRoller("random_roll", window_len=seg_length, min_jump= int(seg_length // 2))

traj_ex = extractor.TrajectoryExtractor(include_config, roller)


# to ensure vehicle ID is unique and identifiable across different routes and scenes
id_generator = extractor.SequentialIDGenerator(start_id= 1e5*route + 1e4*scene)

traj_ex.retrieve_all(ex, filter_dict, id_generator=id_generator)

## Other Datasets (Segmenting to Equi-length Dataset)

In [35]:
import pandas as pd
import dataprocess.schema

HH_FILE_TRAIN = "F:\DATA\HH\data_lyft_HH_CF\data\HH_train.h5"
HH_FILE_VAL = "F:\DATA\HH\data_lyft_HH_CF\data\HH_val.h5"
HH_Col = dataprocess.schema.HH_Col

try:
    df_train = pd.read_hdf(HH_FILE_TRAIN)
    df_val = pd.read_hdf(HH_FILE_VAL)
    print("successfully read HDF5 files.")
    print(df_train.head())
except:
    print("failed to read HDF5 files.")



successfully read HDF5 files.
   case_id  time  x_leader  x_follower  v_leader  v_follower  a_leader  \
0        0   0.0  0.000000  -12.717942  4.183220    1.396880  1.946270   
1        0   0.1  0.545917  -12.731980  4.416340    1.535397  1.944384   
2        0   0.2  1.062604  -12.603732  4.568766    1.372667  1.943686   
3        0   0.3  1.607011  -12.466701  4.753522    1.306147  1.944021   
4        0   0.4  2.177451  -12.303733  4.953015    1.337162  1.945415   

   a_follower  l_leader  l_follower  
0    0.285776  5.014427    4.319781  
1    0.336690  5.014427    4.319781  
2    0.395086  5.014427    4.319781  
3    0.693090  5.014427    4.319781  
4    1.143289  5.014427    4.319781  


In [ ]:
grouped = df_train.groupby(HH_Col.ID)
all_windows = []

for case_id, group in grouped:
    group_values = group.drop(columns=[HH_Col.ID]).to_numpy()  
    
    windows = cut_retrieve_data(group_values, roller=roller)
    all_windows.extend(windows)

return np.array(all_windows)

NameError: name 'cut_retrieve_data' is not defined